# Bert con reglas

In [1]:
import pandas as pd
import numpy as np
import re
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
from tqdm import tqdm

In [2]:
df = pd.read_csv ("../../Datasets/evaluacion2.csv")

In [3]:
display(df.head())
display(df.info())

,product_title,product_rating,is_best_seller,is_sponsored,buy_box_availability,sustainability_tags,has_coupon,discount_percentage,product_category,product_segment,log_original_price,log_purchased_last_month,log_total_reviews
0,OWC 2.0TB Aura Pro X2 (GEN 4) SSD Complete Upg...,4.4,No Badge,Sponsored,1,0,0,0.0,PC Components,Media,5.484755,0.000000,5.883322
1,OWC 250GB Aura Pro 6G Flash SSD Upgrade for 20...,4.6,No Badge,Sponsored,1,0,0,0.0,Storage & Memory Cards,Media,3.783962,0.000000,5.624018
2,HP 67XL Black High-yield Ink Cartridge | Works...,4.6,Best Seller,Organic,0,1,0,0.0,"Office Supplies, Ink & Toner",Media,3.607941,10.819798,11.523598
3,HP 67 Black/Tri-color Ink Cartridges for HP Pr...,4.6,No Badge,Organic,0,0,0,0.0,"Office Supplies, Ink & Toner",Media,3.804215,10.819798,10.986868
4,"Sony ZX Series Wired On-Ear Headphones, Black ...",4.5,Best Seller,Organic,0,0,0,0.0,"Audio, Sound & Recording Gear",Baja,2.355178,9.210440,11.598433


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7717 entries, 0 to 7716
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   product_title             7717 non-null   object 
 1   product_rating            7717 non-null   float64
 2   is_best_seller            7717 non-null   object 
 3   is_sponsored              7717 non-null   object 
 4   buy_box_availability      7717 non-null   int64  
 5   sustainability_tags       7717 non-null   int64  
 6   has_coupon                7717 non-null   int64  
 7   discount_percentage       7717 non-null   float64
 8   product_category          7717 non-null   object 
 9   product_segment           7717 non-null   object 
 10  log_original_price        7717 non-null   float64
 11  log_purchased_last_month  7717 non-null   float64
 12  log_total_reviews         7717 non-null   float64
dtypes: float64(5), int64(3), object(5)
memory usage: 783.9+ KB


None

In [4]:
print("Cargando modelo BERT NER...")
print("(Esto puede tardar 1-2 minutos la primera vez)\n")

model_name = "dslim/bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

ner_pipeline = pipeline(
    "ner", 
    model=model, 
    tokenizer=tokenizer, 
    aggregation_strategy="simple"
)

print("✓ Modelo NER cargado correctamente")

Cargando modelo BERT NER...
(Esto puede tardar 1-2 minutos la primera vez)



Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


✓ Modelo NER cargado correctamente


In [5]:
# Lista de marcas comunes en Amazon (expandible)
COMMON_BRANDS = {
    # Computadoras y Laptops
    'HP', 'Dell', 'Lenovo', 'Apple', 'Asus', 'ASUS', 'Acer', 'MSI', 
    'Microsoft', 'Razer', 'Alienware', 'Huawei', 'LG', 'Gigabyte',
    
    # Componentes PC
    'Intel', 'AMD', 'NVIDIA', 'Corsair', 'EVGA', 'Thermaltake', 'Cooler Master',
    'NZXT', 'be quiet!', 'Fractal Design', 'Lian Li', 'Antec', 'G.Skill',
    
    # Almacenamiento
    'Western Digital', 'Seagate', 'SanDisk', 'Kingston', 'Crucial', 
    'Samsung', 'Transcend', 'PNY', 'Sabrent', 'OWC', 'G-Technology',
    'LaCie', 'Toshiba', 'SK hynix', 'Micron', 'ADATA',
    
    # Periféricos Gaming y General
    'Logitech', 'Razer', 'SteelSeries', 'HyperX', 'Corsair', 'Roccat',
    'Glorious', 'Finalmouse', 'Zowie', 'Cooler Master', 'Redragon',
    '8Bitdo', '8BitDo', 'iFixit',
    
    # Audio
    'Sony', 'Bose', 'JBL', 'Sennheiser', 'Audio-Technica', 'Beyerdynamic',
    'Shure', 'AKG', 'Focal', 'Yamaha', 'Klipsch', 'Polk Audio',
    'Denon', 'Onkyo', 'Marantz', 'KICKER', 'Pioneer', 'Kenwood',
    'Edifier', 'Creative', 'Blue Microphones', 'Rode', 'HyperX',
    'soundcore', 'iFi',
    
    # Monitores y TV
    'Samsung', 'LG', 'Dell', 'ASUS', 'Acer', 'BenQ', 'ViewSonic',
    'AOC', 'Gigabyte', 'MSI', 'VIZIO', 'TCL', 'Hisense', 'Sony',
    'Philips', 'Sceptre', 'Insignia', 'Westinghouse',
    
    # Impresoras y Oficina
    'HP', 'Canon', 'Epson', 'Brother', 'Xerox', 'Ricoh', 'Lexmark',
    'Kodak', 'Fujitsu', 'Plustek', '3M',
    
    # Fotografía
    'Canon', 'Nikon', 'Sony', 'Fujifilm', 'Panasonic', 'Olympus',
    'Pentax', 'Leica', 'GoPro', 'DJI', 'Insta360', 'Sigma', 'Tamron',
    
    # Redes y Conectividad
    'TP-Link', 'Netgear', 'Linksys', 'ASUS', 'Ubiquiti', 'D-Link',
    'Belkin', 'Aruba', 'Cisco', 'Google', 'eero', 'Synology', 'QNAP',
    'UniFi', 'MikroTik', 'Zyxel', 'TRENDnet', 'weBoost', 'WAVLINK',
    
    # Streaming y Media
    'Roku', 'Amazon', 'Google', 'Apple', 'Fire TV', 'Chromecast',
    'NVIDIA Shield', 'Plex',
    
    # Smartphones y Tablets
    'Apple', 'Samsung', 'Google', 'OnePlus', 'Xiaomi', 'Motorola',
    'Nokia', 'Huawei', 'OPPO', 'Vivo', 'Realme', 'Asus', 'Lenovo',
    
    # Accesorios y Cables
    'Anker', 'Belkin', 'AmazonBasics', 'Cable Matters', 'Monoprice',
    'Ugreen', 'Aukey', 'RAVPower', 'Spigen', 'OtterBox', 'Case-Mate',
    'UAG', 'Dbrand', 'JSAUX', 'UGREEN', 'iOttie', 'iHome',
    'kate spade', 'PopSockets',
    
    # Smart Home
    'Ring', 'Nest', 'Arlo', 'Wyze', 'Philips Hue', 'LIFX', 'Ecobee',
    'August', 'Lutron', 'TP-Link Kasa', 'Eufy', 'Blink', 'SimpliSafe',
    'myQ',
    
    # Seguridad y Cámaras
    'Lorex', 'Swann', 'Reolink', 'Amcrest', 'Hikvision', 'Ring',
    'Arlo', 'Nest', 'Wyze', 'Blink', 'Eufy',
    
    # Garantías Extendidas
    'ASURION', 'SquareTrade', 'Allstate', 'Extend',
    
    # Marcas Amazon
    'AmazonBasics', 'Amazon Essentials', 'Eero', 'Ring', 'Blink',
    'Fire TV', 'Kindle', 'Echo',
    
    # Baterías y Energía
    'Duracell', 'Energizer', 'Panasonic', 'Maxell', 'Renata',
    
    # Ropa deportiva y accesorios
    'adidas', 'Nike', 'Under Armour', 'Puma', 'New Balance',
    
    # Gaming y Nintendo
    'amiibo', 'Nintendo', 'PlayStation', 'Xbox',
    
    # Otras marcas tech importantes
    'Wacom', 'Elgato', 'Blue Yeti', 'Focusrite', 'Behringer',
    'PreSonus', 'M-Audio', 'Novation', 'Native Instruments',
    'Blackmagic', 'AVerMedia', 'Streamlabs', 'NZXT', 'Thermaltake'
}

def extract_brand_hybrid(title):
    """
    Extrae marca combinando reglas heurísticas y NER
    
    Estrategia:
    1. Buscar marcas conocidas en las primeras palabras
    2. Aplicar reglas de patrones comunes en Amazon
    3. Usar NER como fallback
    """
    if pd.isna(title) or title == "":
        return None
    
    # REGLA 1: Buscar marcas conocidas en las primeras 4 palabras
    words = title.split()
    first_words = ' '.join(words[:4])
    
    for brand in COMMON_BRANDS:
        # Buscar coincidencia exacta (case-insensitive)
        pattern = r'\b' + re.escape(brand) + r'\b'
        if re.search(pattern, first_words, re.IGNORECASE):
            return brand
    
    # REGLA 2: La marca suele estar al principio antes de números/símbolos
    # Patrón: Captura palabras en mayúscula al inicio hasta encontrar:
    # - Un número
    # - Símbolos como |, -, (, [
    # - Palabras comunes que no son marcas (for, with, in, etc.)
    match = re.match(r'^([A-Z][A-Za-z0-9]*(?:\s+[A-Z&][A-Za-z0-9]*)*?)(?:\s+\d|\s*[\|\-\(,\[]|$|\s+(?:for|with|in|by|and|the|a|an)\s)', title)
    
    if match:
        potential_brand = match.group(1).strip()
        
        # Validaciones:
        # - No más de 3 palabras
        # - No palabras genéricas
        generic_words = {'New', 'Used', 'Refurbished', 'Original', 'Genuine', 'Compatible'}
        
        if len(potential_brand.split()) <= 3 and potential_brand not in generic_words:
            return potential_brand
    
    # REGLA 3: Si la primera palabra empieza con mayúscula, probablemente sea la marca
    if words and words[0][0].isupper() and len(words[0]) > 1:
        # Tomar palabras consecutivas en mayúscula
        brand_parts = []
        for word in words[:3]:  # Máximo 3 palabras
            # Palabras en mayúscula o conectores
            if word[0].isupper() or word.lower() in ['&', 'and']:
                brand_parts.append(word)
            else:
                break
        
        if brand_parts:
            potential = ' '.join(brand_parts)
            # No retornar si es muy genérico
            if potential.lower() not in ['new', 'used', 'refurbished', 'original']:
                return potential
    
    # FALLBACK: Usar NER (solo si tiene alta confianza)
    try:
        entities = ner_pipeline(title)
        for entity in entities:
            # Solo ORG cerca del inicio y con confianza alta
            if (entity['entity_group'] == 'ORG' 
                and entity['start'] < 50 
                and entity['score'] > 0.85
                and len(entity['word'].strip()) > 1):
                return entity['word'].strip()
    except:
        pass
    
    return None

print("✓ Función híbrida definida")

✓ Función híbrida definida


In [6]:
# Probar con 50 productos aleatorios
print("Probando con 50 productos aleatorios:\n" + "="*80 + "\n")

sample = df.sample(50, random_state=42)

# Contadores para estadísticas
detectadas = 0
no_detectadas = 0

for idx, row in sample.iterrows():
    title = row['product_title']
    brand = extract_brand_hybrid(title)
    
    title_short = title[:65] + "..." if len(title) > 65 else title
    
    if brand:
        print(f"✓ {title_short}")
        print(f"  → Marca: {brand}\n")
        detectadas += 1
    else:
        print(f"✗ {title_short}")
        print(f"  → Marca: ❌ No detectada\n")
        no_detectadas += 1

# Resumen
print("="*80)
print("RESUMEN DE LA MUESTRA (50 productos)")
print("="*80)
print(f"Marcas detectadas:     {detectadas} ({detectadas/50*100:.1f}%)")
print(f"Marcas NO detectadas:  {no_detectadas} ({no_detectadas/50*100:.1f}%)")
print("="*80)

Probando con 50 productos aleatorios:

✓ ASURION 2 Year Wearables Protection Plan ($250 - $299.99)
  → Marca: ASURION

✓ Brother Genuine High Yield Toner Cartridge, TN850, Replacement Bl...
  → Marca: Brother

✓ KICKER CS Series CSC68 6 x 8 Inch Car Audio System Speaker, Black...
  → Marca: KICKER

✓ Roku Voice Remote Pro | Rechargeable with Hands-free Voice Contro...
  → Marca: Roku

✓ Acer Nitro 16 Gaming Laptop | AMD Ryzen 9 7940HS Octa-Core CPU | ...
  → Marca: Acer

✓ Klipsch AW-650 Indoor/Outdoor Speaker, White (Pair)
  → Marca: Klipsch

✓ HP USB-C Dock G5 (Renewed)
  → Marca: HP

✓ Samsung 27" Odyssey QHD G65B Curved Gaming Monitor, 240Hz, 1ms (G...
  → Marca: Samsung

✓ Razer Cobra Gaming Mouse: 58g, Gen-3 Optical Switches, Chroma RGB...
  → Marca: Razer

✓ TP-Link TL-SG3452P | 48 Port Gigabit L2+ Managed PoE Switch | 48 ...
  → Marca: TP-Link

✓ Insta360 X4-8K Waterproof 360 Action Camera, 4K Wide-Angle Video,...
  → Marca: Insta360

✓ Baofeng Walkie Talkies 888S Rechargeable 

In [7]:
print("Aplicando extracción a TODO el dataset...")

brands = []
for title in tqdm(df['product_title'], desc="Extrayendo marcas", ncols=100):
    brands.append(extract_brand_hybrid(title))

df['brand'] = brands

print("\n✓ Extracción completada\n")

# Estadísticas finales
total = len(df)
con_marca = df['brand'].notna().sum()
sin_marca = total - con_marca
cobertura = (con_marca / total) * 100

print("="*70)
print("RESULTADOS FINALES")
print("="*70)
print(f"Total productos:      {total:,}")
print(f"Con marca detectada:  {con_marca:,} ({cobertura:.1f}%)")
print(f"Sin marca:            {sin_marca:,} ({100-cobertura:.1f}%)")
print(f"Marcas únicas:        {df['brand'].nunique()}")
print("="*70)

Aplicando extracción a TODO el dataset...


Extrayendo marcas: 100%|██████████████████████████████████████| 7717/7717 [00:04<00:00, 1829.25it/s]


✓ Extracción completada

RESULTADOS FINALES
Total productos:      7,717
Con marca detectada:  7,700 (99.8%)
Sin marca:            17 (0.2%)
Marcas únicas:        1752


In [8]:
# Ruta de salida
output_path = "../../Datasets/evaluacion_brand_bert_hibrido.csv"

# Guardar dataframe final con la columna 'brand'
df.to_csv(output_path, index=False, encoding="utf-8")

print(f"✓ Dataset guardado correctamente en:\n{output_path}")


✓ Dataset guardado correctamente en:
../../Datasets/evaluacion_brand_bert_hibrido.csv
